# AdaRoute Quickstart (Google Colab & Anaconda)
**Threshold-Calibrated Decision-Entropy Routing and Bayesian Virtual Evidence Pooling**

This notebook demonstrates `adaroute` end-to-end in Google Colab or Anaconda Jupyter.

In [1]:
import numpy as np
from adaroute import (
    AdaRouter,
    LogLinearOpinionPooler,
    MemoizedVariableElimination,
    adaroute_decision_entropy,
    shannon_entropy,
)

# 1. Compare raw Shannon entropy H(p) vs. threshold-calibrated H_{tau*}(p) at tau* = 0.0267
tau_star = 0.0267
probs = np.array([0.002, tau_star, 0.50])
print('Raw Shannon H(p):      ', np.round(shannon_entropy(probs), 4))
print('AdaRoute H_{tau*}(p):  ', np.round(adaroute_decision_entropy(probs, tau_star=tau_star), 4))

In [2]:
# 2. Route trajectories across SKIP_SYSTEM1_CONFIDENT, LOW, MEDIUM, and HIGH
router = AdaRouter(tau_star=tau_star, scale=0.55)
tiers = router.route(probs, missing_counts=[0, 2, 0])
summary = router.summarize(tiers)
print('Assigned Tiers:', tiers)
print('Token Compute Saved:', f'{summary.estimated_token_savings_fraction:.1%}')

In [3]:
# 3. Fuse System 1 and System 2 posteriors via precision-weighted log-linear pooling
pooler = LogLinearOpinionPooler().fit_threshold_shift(tau_sys1=0.0267, tau_sys2=0.0332)
fused = pooler.fuse(probs, [0.002, 0.140, 0.50], tiers=tiers, thought_tokens=[0, 1540, 0])
print('Fused Posteriors:', np.round(fused, 4))